In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from dreamer import Dreamer
from utils import seedEverything, loadConfig

In [2]:
# Set device
device = torch.device("mps")
print(f"Using device: {device}")

# Set random seed for reproducibility
seed = 42
seedEverything(seed)

Using device: mps


In [3]:
# Load configuration from celtic-heroes.yml
config = loadConfig("celtic-heroes.yml")

In [ ]:
config

In [4]:
# Define observation shape and action size for a simple environment
# Using a small image size for testing
observation_shape = (3, 128, 128)  # (channels, height, width)
action_size = 4  # Number of discrete actions

print(f"Observation shape: {observation_shape}")
print(f"Action size: {action_size}")

# Initialize the Dreamer agent
dreamer = Dreamer(observation_shape, action_size, device, config.dreamer)
print("Dreamer agent initialized successfully!")


Observation shape: (3, 128, 128)
Action size: 4
Dreamer agent initialized successfully!


In [5]:
# Function to generate random observations (images)
def generate_random_observation(shape):
    # Generate random values between 0 and 1
    return np.random.rand(*shape).astype(np.float32)

# Function to generate random actions
def generate_random_action(size):
    # Generate a one-hot encoded action
    action = np.zeros(size, dtype=np.float32)
    action[np.random.randint(0, size)] = 1.0
    return action

# Generate multiple sequences of synthetic transitions with random lengths
num_sequences = 5  # Number of different sequences to generate
total_transitions = 0

for seq in range(num_sequences):
    # Randomly generate number of transitions between 50 and 400
    num_transitions = np.random.randint(50, 401)
    total_transitions += num_transitions

    print(f"Generating sequence {seq+1}/{num_sequences} with {num_transitions} transitions...")
    for i in range(num_transitions):
        # Generate random observation
        obs = generate_random_observation(observation_shape)

        # Generate random action
        action = generate_random_action(action_size)

        # Generate random reward
        reward = np.random.uniform(-1, 1)

        # Generate random next observation
        next_obs = generate_random_observation(observation_shape)

        # Generate random done flag (mostly False, occasionally True)
        done = np.random.random() < 0.05  # 5% chance of episode ending

        # Add transition to buffer
        dreamer.buffer.add(obs, action, reward, next_obs, done)

print(f"Total transitions generated: {total_transitions}")
print(f"Buffer size: {len(dreamer.buffer)}")


Generating sequence 1/5 with 152 transitions...
Generating sequence 2/5 with 206 transitions...
Generating sequence 3/5 with 264 transitions...
Generating sequence 4/5 with 327 transitions...
Generating sequence 5/5 with 314 transitions...
Total transitions generated: 1263
Buffer size: 1263


In [6]:
# Training parameters
num_iterations = 1
world_model_losses = []
actor_losses = []
critic_losses = []

print("Starting training...")
for i in range(num_iterations):
    # Sample data from buffer
    sampled_data = dreamer.buffer.sample(dreamer.config.batchSize, dreamer.config.batchLength)

    # Train world model
    initial_states, world_model_metrics = dreamer.worldModelTraining(sampled_data)

    # Train behavior (actor and critic)
    behavior_metrics = dreamer.behaviorTraining(initial_states)

    # Store metrics for plotting
    world_model_losses.append(world_model_metrics["worldModelLoss"])
    actor_losses.append(behavior_metrics["actorLoss"])
    critic_losses.append(behavior_metrics["criticLoss"])

    # Print metrics every 10 iterations
    if (i + 1) % 10 == 0:
        print(f"Iteration {i+1}/{num_iterations}")
        print(f"World Model Loss: {world_model_metrics['worldModelLoss']:.4f}")
        print(f"Reconstruction Loss: {world_model_metrics['reconstructionLoss']:.4f}")
        print(f"Reward Prediction Loss: {world_model_metrics['rewardPredictorLoss']:.4f}")
        print(f"KL Loss: {world_model_metrics['klLoss']:.4f}")
        print(f"Actor Loss: {behavior_metrics['actorLoss']:.4f}")
        print(f"Critic Loss: {behavior_metrics['criticLoss']:.4f}")
        print(f"Entropy: {behavior_metrics['entropies']:.4f}")
        print("---")

print("Training completed!")


Starting training...
Training completed!


In [ ]:
# Plot training losses
plt.figure(figsize=(12, 8))

# World model loss
plt.subplot(3, 1, 1)
plt.plot(world_model_losses)
plt.title('World Model Loss')
plt.grid(True)

# Actor loss
plt.subplot(3, 1, 2)
plt.plot(actor_losses)
plt.title('Actor Loss')
plt.grid(True)

# Critic loss
plt.subplot(3, 1, 3)
plt.plot(critic_losses)
plt.title('Critic Loss')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Sample a batch of data
test_data = dreamer.buffer.sample(4, dreamer.config.batchLength)

# Get original images
original_images = test_data.observations[:, 0].cpu().numpy()

# Encode and then decode the images
with torch.no_grad():
    # Encode observations
    encoded_obs = dreamer.encoder(test_data.observations[:, 0])

    # Initialize recurrent and latent states
    recurrent_state = torch.zeros(4, dreamer.recurrentSize, device=dreamer.device)

    # Get posterior latent state
    _, posterior_logits = dreamer.posteriorNet(torch.cat((recurrent_state, encoded_obs), -1))
    posterior_dist = torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=posterior_logits), 1)
    latent_state = posterior_dist.rsample().view(-1, dreamer.latentSize)

    # Create full state
    full_state = torch.cat((recurrent_state, latent_state), -1)

    # Decode to get reconstructed images
    reconstructed_images = dreamer.decoder(full_state).cpu().numpy()

# Plot original and reconstructed images
plt.figure(figsize=(12, 8))
for i in range(4):
    # Original image
    plt.subplot(4, 2, 2*i+1)
    plt.imshow(np.transpose(original_images[i], (1, 2, 0)))
    plt.title(f"Original {i+1}")
    plt.axis('off')

    # Reconstructed image
    plt.subplot(4, 2, 2*i+2)
    plt.imshow(np.transpose(reconstructed_images[i], (1, 2, 0)))
    plt.title(f"Reconstructed {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.show()
